# Preliminary lab

We use a tiny coffee-shop transactions dataset.  
Each row is one **receipt** with a unique `ReceiptID` and an `Item`.

You can hard-code this table in Python (or save as CSV if you prefer):

| ReceiptID | Item        |
|----------:|------------|
| 1         | Espresso   |
| 1         | Croissant  |
| 1         | Water      |
| 2         | Latte      |
| 2         | Muffin     |
| 3         | Espresso   |
| 3         | Latte      |
| 3         | Muffin     |
| 4         | Cappuccino |
| 4         | Croissant  |
| 5         | Espresso   |
| 5         | Cappuccino |
| 5         | Brownie    |
| 6         | Latte      |
| 6         | Croissant  |
| 6         | Water      |
| 7         | Espresso   |
| 7         | Muffin     |
| 8         | Cappuccino |
| 8         | Muffin     |
| 8         | Brownie    |
| 9         | Latte      |
| 9         | Brownie    |
| 10        | Espresso   |
| 10        | Latte      |
| 10        | Croissant  |
| 10        | Muffin     |

Items:  
`Espresso, Latte, Cappuccino, Croissant, Muffin, Brownie, Water`


#### Task 1.1 – Load as a pandas DataFrame

1. Manually create this dataset as a `pandas.DataFrame` with columns:
   - `ReceiptID` (int)
   - `Item` (string)
2. Print:
   - first 5 rows,
   - number of unique receipts,
   - number of unique items.


In [ ]:
import pandas as pd

import numpy as np

from IPython.display import display

from sklearn.preprocessing import OneHotEncoder

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

In [ ]:
# don't ask why it's txt but the read_scv still works ....
df = pd.read_csv('../../Dataset/LAB7/toy_dataset.txt', sep=',')

display(df)

In [ ]:
df.rename(columns={'ReceiptID': 'id', 'Item': 'item'}, inplace=True)

In [ ]:
display(df.head(5))

In [ ]:
print(df['id'].value_counts())

In [ ]:
print(df['item'].value_counts())

#### Task 1.2 – From long format to “transaction list”

Build a Python object that maps each `ReceiptID` to the set (or list) of items:

```python
transactions = {
    1: ['Espresso', 'Croissant', 'Water'],
    2: ['Latte', 'Muffin'],
    ...
}
```

In [ ]:
grouped = df.groupby('id')

transactions = {}

for id, values in grouped:
    # print(f"id: {id}")
    # print(f"{values['item']}")
    # print()

    transactions[id] = values['item'].tolist()

print(transactions)


#### Task 1.3 – One-hot encoding (basket → 0/1 matrix)

For most Apriori / FP-growth implementations in Python, you need a one-hot encoded DataFrame:  
	•	Rows = transactions (receipts)  
	•	Columns = items  
	•	Cell = 1 if item is present in that transaction, 0 otherwise  

Like this:
| ReceiptID | Espresso | Latte | Cappuccino | Croissant | Muffin | Brownie | Water |
|-----------|----------|-------|------------|-----------|--------|---------|-------|
| 1         | 1        | 0     | 0          | 1         | 0      | 0       | 1     |
| 2         | 0        | 1     | 0          | 0         | 1      | 0       | 0     |

> OneHotEncode the grouped DS --> pd.get_dummies()  
**Actually first encode and then group**

In [ ]:
# I cannot one hot encode transactions because the values associated to some keys have different lenghts --> OHE the original DS and then groupby 
OHE_df = pd.get_dummies(df)
# display(OHE_df)

# grouped_OHE_df = OHE_df.groupby('id')
    # .any()
    # 	•	[False, False, True] → True
    # 	•	[False, False, False] → False

grouped_OHE_df = OHE_df.groupby('id').any()
display(grouped_OHE_df)

# instead of True and False use 1/0
d = {}
for col in grouped_OHE_df:
    l = []
    for value in grouped_OHE_df[col]:
        if value == False:
            v = 0
            l.append(v)
        elif value == True:
            v = 1
            l.append(v)
    d[col] = l

compact_OHE_DF = pd.DataFrame(d, index=grouped_OHE_df.index)
display(compact_OHE_DF)



#### Part 2 – Use a library Apriori / FP-growth

> **Now we use a library to get frequent itemsets and association rules on the same small dataset.**

You can use, for example:
- mlxtend.frequent_patterns.apriori  
- mlxtend.frequent_patterns.fpgrowth  
- mlxtend.frequent_patterns.association_rules  

#### Task 2.1 – Run Apriori on the coffee-shop data
1.	Use the one-hot encoded DataFrame as input to apriori (or analogous function).  
2.	Choose a small min_support (e.g. 0.2 or 0.3, up to you).  
3.	Obtain a DataFrame of frequent itemsets with their support.  
4.	Inspect:  
- How many 1-itemsets are frequent?
- How many 2-itemsets?
- Are there any 3-itemsets? (If not, try lowering min_support.)

Do not hard-code expected numbers; just observe and note what you see.

# HOW DOES THE APRIORI WORKS?

#### Generating Frequent Itemsets
> The **apriori** function **expects data in a one-hot encoded pandas DataFrame**  
Example:
```python
dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]
```

> We can transform it into the right format via the **TransactionEncoder** --> NO FUCKING WAY ANOTHER ENCODER
```python
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)
```

Then we recall apriori like this:  
```python
apriori(df, min_support=0.6)
```

Which returns something like this:
```text
support	itemsets
0	0.8	(3)
1	1.0	(5)
```

By default, apriori returns the column indices of the items --> BUT we can set use_colnames=True to convert these integer values into the respective item names:
```python
apriori(df, min_support=0.6, use_colnames=True)
```
Which returns:
```text
support	itemsets
0	0.8	(Eggs)
1	1.0	(Kidney Beans)
```

####  Selecting and Filtering Results
The advantage of working with pandas DataFrames is that we can use its convenient features to filter the results. For instance, let's assume we are only interested in itemsets of length 2 that have a support of at least 80 percent. First, we create the frequent itemsets via apriori and add a new column that stores the length of each itemset:
```python
frequent_itemsets = apriori(df, min_support=0.6, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(lambda x: len(x))
```

- Returns:

```text
support	itemsets	length
0	0.8	(Eggs)	1
1	1.0	(Kidney Beans)	1
```

- Then, we can select the results that satisfy our desired criteria as follows:
```python
frequent_itemsets[ (frequent_itemsets['length'] == 2) &
                   (frequent_itemsets['support'] >= 0.8) ]
```

- Returns:

```text
support	itemsets	length
5	0.8	(Eggs, Kidney Beans)	2
```

In [ ]:
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

In [ ]:
display(OHE_df)

In [ ]:
display(grouped_OHE_df)

In [ ]:
# frequent_itemsets = apriori(OHE_df, min_support=0.3)
# --> doesn't work as id is a column, not index --> apriori expects only True, False, 0 or 1,
# BUT since id is a column id also has values 2,3,4,5,6,... so it crashes --> grouped_OHE_df has id as index, so no problems

frequent_itemsets = apriori(grouped_OHE_df, min_support=0.2, use_colnames=True)
# --> an itemset must appear in at least 20% of all transactions!

# a = sorted(frequent_itemsets['support'], key=lambda x:x, reverse=True ) --> get a plain python list, loose DF
frequent_itemsets = frequent_itemsets.sort_values(by='support', ascending=False)
frequent_itemsets = frequent_itemsets.sort_values(by='itemsets', ascending=False)
display(frequent_itemsets)

#### Task 2.2 – Run FP-growth on the same data
1.	Run the FP-growth implementation (if available) on the same one-hot encoded DataFrame and same min_support.  
2.	Compare:  
- Are the frequent itemsets identical to those from Apriori (up to ordering)?
- Are the supports identical (up to floating point noise)?

Write a short comment in code or markdown: “Apriori vs FP-growth: what seems different? What’s the same?”  

In [ ]:
# identical as apriori, just different algorithm: FP-growth

frequent_itemsets_2 = fpgrowth(grouped_OHE_df, min_support=0.2, use_colnames=True)
frequent_itemsets_2 = frequent_itemsets_2.sort_values(by = 'support', ascending=False)
frequent_itemsets_2 = frequent_itemsets_2.sort_values(by = 'itemsets', ascending=False)
# display(frequent_itemsets_2)

    
# get first element of frequent_itemsets,
# search it in frequent_itemsets_2
# get its support
# compare supports, if identical increase count

count = 0
for i in range(len(frequent_itemsets['itemsets'])):
    # fix row i in frequent_itemsets
    itemset_1 = frequent_itemsets['itemsets'][i]
    support_1 = frequent_itemsets['support'][i]

    # search the same itemset in frequent_itemsets_2
    for j in range(len(frequent_itemsets_2['itemsets'])):
        if frequent_itemsets_2['itemsets'][j] == itemset_1:
            support_2 = frequent_itemsets_2['support'][j]

            # compare supports
            if support_1 == support_2:
                count += 1

            # stop searching this itemset_1 in frequent_itemsets_2
            break

print(f"Number of itemsets with identical support in both: {count} \ {len(frequent_itemsets['itemsets'])}")

Task 2.3 – Generate association rules

Using the frequent itemsets from either Apriori or FP-growth:
1.	Generate association rules with:  
- metric: "confidence" (or "lift"),
- min_threshold of your choice (e.g. 0.6).

```python
rules = association_rules()
```

### Association rule in human words

A rule is something like:

> `{Diapers} → {Beer}`  

Read it as:

> “When people buy diapers, they often also buy beer.”

Left side = **antecedent** (if…)  
Right side = **consequent** (…then often this).

No magic, no causality, just “these things appear together a lot in the data”.


### SUPPORT – “How common is this combo overall?”

For a rule `A → C`:

- **support(A→C)** = fraction of all receipts that contain **both** A and C.

So if you have 100 receipts and 7 of them have both Diapers and Beer:

> support({Diapers}→{Beer}) = 7 / 100 = 0.07  

Range: \([0,1]\).  
If support is tiny, it means “this thing almost never happens”, even if the rule looks cool.


### CONFIDENCE – “Given A, how often do we also see C?”

For a rule `A → C`:

$\text{confidence}(A \to C) = \frac{\text{support}(A \cup C)}{\text{support}(A)}$

Interpretation:

> Among the receipts that contain A, what fraction also contain C?

Example:

- 20 receipts have Diapers.
- 10 receipts have both Diapers and Beer.

Then:

> confidence({Diapers}→{Beer}) = 10 / 20 = 0.5  

So: “In 50% of diaper receipts, there is also beer.”

Range: \([0,1]\).



### LIFT – “Is this really a pattern, or just because C is popular anyway?” --> meh

Lift compares the rule to “random chance”.

$\text{lift}(A \to C) = \frac{\text{confidence}(A \to C)}{\text{support}(C)}$

- If **lift = 1** → A and C are **independent**. Knowing A doesn’t change how likely C is.
- If **lift > 1** → A and C happen together **more often than chance** → interesting.
- If **lift < 1** → A kind of “avoids” C.

Example:

- confidence({Diapers}→{Beer}) = 0.5  
- support({Beer}) = 0.25  

Then:

> lift = 0.5 / 0.25 = 2  

So: “People with diapers buy beer **twice as often** as we’d expect if diapers and beer had nothing to do with each other.”

---
**Example:**
```python
dataset = [['Milk', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Dill', 'Onion', 'Nutmeg', 'Kidney Beans', 'Eggs', 'Yogurt'],
           ['Milk', 'Apple', 'Kidney Beans', 'Eggs'],
           ['Milk', 'Unicorn', 'Corn', 'Kidney Beans', 'Yogurt'],
           ['Corn', 'Onion', 'Onion', 'Kidney Beans', 'Ice cream', 'Eggs']]

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

frequent_itemsets = fpgrowth(df, min_support=0.6, use_colnames=True)
```

- As we did
- The **generate_rules()** function allows you to (1) specify your metric of interest and (2) the according threshold. Currently implemented measures are confidence and lift. Let's say you are interested in rules derived from the frequent itemsets only if the level of confidence is above the 70 percent threshold (min_threshold=0.7):

```python
association_rules(frequent_itemsets, metric="confidence", min_threshold=0.7, num_itemsets=len(df.index))
```

> association_rules(  
    **from who are you trying to figure out association rules?**,  
    metric="**on what matric do you want to extract these association rules?**",  
    min_threshold= **out of all frequent itemset on which you're trying to figure out the association rules,  
    out of these what's the minimum frequency these frequent item must have?**,  
    num_itemsets= **how many items are we processing and trying ot extract association rule from?**  
    )  


- Output:

| antecedents        | consequents       | antecedent support | consequent support | support | confidence | lift | representativity | leverage |
|--------------------|-------------------|---------------------|--------------------|---------|------------|------|------------------|----------|
| (Kidney Beans)     | (Eggs)            | 1.0                 | 0.8                | 0.8     | 0.80       | 1.00 | 1.0              | 0.00     |
| (Eggs)             | (Kidney Beans)    | 0.8                 | 1.0                | 0.8     | 1.00       | 1.00 | 1.0              | 0.00     |
| (Yogurt)           | (Kidney Beans)    | 0.6                 | 1.0                | 0.6     | 1.00       | 1.00 | 1.0              | 0.00     |


- How to read this
    - Antecedent Support:
	    - This is the proportion of transactions that contain the antecedent itemset.
	    - For the first row, the antecedent (Kidney Beans) appears in 100% of the transactions (support = 1.0).
    
    - Consequent Support:
        - This is the proportion of transactions that contain the consequent itemset.
        - For the first row, the consequent (Eggs) appears in 80% of the transactions (support = 0.8).
    
    - Support:
        - The support for the rule is the proportion of transactions that contain both the antecedent and the consequent.
        - In the first row, support = 0.8 means that 80% of transactions contain both Kidney Beans and Eggs.
    
    - Confidence:
        - This is the probability that the consequent is bought given that the antecedent was bought.
        - In the first row, confidence = 0.80, meaning that 80% of transactions with Kidney Beans also contain Eggs.


---


2.	Inspect at least 5 rules:  
- antecedents (left side),
- consequents (right side),
- support,
- confidence,
- lift (if available).

For each of 2–3 rules, answer in words:
- “What does this rule say in plain language?”
- “Is it intuitive on this tiny coffee-shop dataset?”

No numbers need to be perfect; the main goal is to read & interpret rules.

In [ ]:
# Task 2.3 – Generate association rules
# Using the frequent itemsets from either Apriori or FP-growth:
# 1. Generate association rules with:  
# - metric: "confidence" (or "lift"),
# - min_threshold of your choice (e.g. 0.6).

# association_rules

AS = association_rules(frequent_itemsets, metric = 'confidence', min_threshold=0.65, num_itemsets=len(grouped_OHE_df.index))
display(AS)


#### Part 3 – Hand-coded Apriori on the same tiny dataset

> Now you implement a very simple version of Apriori yourself, only on this tiny dataset.
> You do not have to optimize; clarity > efficiency.


#### Task 3.1 – Support counting helper

Write a helper function:
```python
def compute_support(itemset, transactions):
    """
    itemset: a Python set or frozenset of items, e.g. {'Espresso', 'Muffin'}
    transactions: dict or list of itemsets (like your `transactions` dict)
    returns: support as a fraction in [0,1]
    """
    ...
```

Requirements:  
	•	Count in how many receipts the itemset appears as a subset.  
	•	Divide by total number of receipts.  
	•	Test it on a few simple itemsets:  
	    •	singletons like {Espresso}  
	    •	pairs like {Espresso, Muffin}  

Do not print the “correct” support explicitly in the lab text; just test and verify in your code.  

In [ ]:
# {1: ['Espresso', 'Croissant', 'Water'], 2: ['Latte', 'Muffin'], 3: ['Espresso', 'Latte', 'Muffin'], 4: ['Cappuccino', 'Croissant'], 5: ['Espresso', 'Cappuccino', 'Brownie'], 6: ['Latte', 'Croissant', 'Water'], 7: ['Espresso', 'Muffin'], 8: ['Cappuccino', 'Muffin', 'Brownie'], 9: ['Latte', 'Brownie'], 10: ['Espresso', 'Latte', 'Croissant', 'Muffin']}

def compute_support(itemset, transactions):
    """
    This function is supposed to compute the support for a given itemset, so:
    itemset: a Python set or frozenset of items, e.g. {'Espresso', 'Muffin'}
    transactions: dict or list of itemsets, all receipts (like your `transactions` dict)
    Computes frequency of itemset among all receipts / all receipts
    returns: support as a fraction in [0,1]
    """
    
    count = 0
    for receipt in transactions:
        found = True

        for item in itemset:
            
            if transactions[receipt].count(item) >= 1:
                continue
            else:
                found = False
                break

        if found:
            count += 1

    
    # support = occurencies item / all receits
    support = count / len(transactions)
    
    return support

if __name__ == '__main__':
    itemset = {'Espresso', 'Muffin'}
    itemset_2 = {'Espresso'}
    transactions = {1: ['Espresso', 'Croissant', 'Water'], 2: ['Latte', 'Muffin'], 3: ['Espresso', 'Latte', 'Muffin'], 4: ['Cappuccino', 'Croissant'], 5: ['Espresso', 'Cappuccino', 'Brownie'], 6: ['Latte', 'Croissant', 'Water'], 7: ['Espresso', 'Muffin'], 8: ['Cappuccino', 'Muffin', 'Brownie'], 9: ['Latte', 'Brownie'], 10: ['Espresso', 'Latte', 'Croissant', 'Muffin']}

    support = compute_support(itemset_2, transactions)
    print(f"The itemset: {itemset} \nHas support: {support}")


Task 3.2 – Generate candidate k-itemsets

Implement:
```python
def generate_candidates(prev_freq_itemsets, k):
    """
    prev_freq_itemsets: list of frequent (k-1)-itemsets (as sets/frozensets)
    k: size of new candidate itemsets to generate
    returns: list (or set) of candidate k-itemsets
    """
    ...
```
**Example**  
Say at level k−1 = 1 you have found these frequent 1-itemsets:  
```python
prev_freq_itemsets = [
    {'A'},
    {'B'},
    {'C'}
]
k = 2
```

generate_candidates(prev_freq_itemsets, k=2) should return all possible 2-itemsets built from them:  
- {A, B}  
- {A, C}  
- {B, C}  

So the result is something like:  
```python
candidates_2 = [
    frozenset({'A', 'B'}),
    frozenset({'A', 'C'}),
    frozenset({'B', 'C'}),
]
```

Those are the candidate 2-itemsets C2.  
Later, you’ll scan the DB and compute their support to decide which ones are frequent (L2).  


Hints (no full solution):  
- Use join: combine pairs of (k−1)-itemsets that share the same first k−2 items (after sorting).  
- Use prune: for each candidate of size k, check that all its (k−1) subsets are in prev_freq_itemsets.  
- Represent itemsets as frozenset to allow usage in sets/dicts.  

In [ ]:
def generate_candidates(prev_freq_itemsets, k):
    """
    prev_freq_itemsets: list of frequent (k-1)-itemsets (as sets/frozensets)
    k: size of new candidate itemsets to generate
    returns: list (or set) of candidate k-itemsets
    """
    new_candidates = []
    for i in range(len(prev_freq_itemsets)):
        for j in range(i + 1, len(prev_freq_itemsets)):

            fixed_candidate = prev_freq_itemsets[i]
            varying_candidate = prev_freq_itemsets[j]

            # concatenate them --> thois weird way is how you concatenate frozenset ... bah
            new_candidate = fixed_candidate | varying_candidate

            if len(new_candidate) == k:
                new_candidates.append(sorted(new_candidate))

    return new_candidates

    


if __name__ == '__main__':
    prev_freq_itemsets = [frozenset({'A', 'B'}),
                          frozenset({'B'}),
                          frozenset({'C'}),
                          frozenset({'D'})]
    k = 2

    new_candidates = generate_candidates(prev_freq_itemsets, k)
    #  {'A', 'B'}, {'A', 'C'}, {'A', 'D'}, {'B', 'C'}, {'B', 'D'}, {'C', 'D'}

    print(new_candidates)
    

Task 3.3 – One full Apriori pass (k = 1, 2, 3)

Implement a minimal Apriori loop:
1.	Construct L1:  
- generate all 1-itemsets,  
- compute their support,  
- keep those with support ≥ minsup (e.g. 0.2).  

2.	Generate C2 from L1, compute support, prune to get L2.  
3.	Generate C3 from L2, compute support, prune to get L3.  
4.	Stop when Lk becomes empty.  

Tasks:  
	•	Print L1, L2, L3 (itemsets + support).  
	•	Compare qualitatively (not numerically) with the library Apriori results:  
	•	Do you see similar frequent patterns?  
	•	Are the “strong” itemsets (very frequent ones) the same?  

#### I mean this kinda works but it's horrendous, I'm manually doing the algorithm

In [ ]:
def get_data(file_path):
    df = pd.read_csv(file_path, sep=',')
    df.rename(columns={'ReceiptID': 'id', 'Item': 'item'}, inplace=True)
    return df
    
def get_transactions(df):
    
    grouped = df.groupby('id')
    transactions = {}
    for id, values in grouped:
        # print(f"id: {id}")
        # print(f"{values['item']}")
        # print()

        transactions[id] = values['item'].tolist()
    
    return transactions



def get_f_1_itemsets(transactions, minsup=0.3):
    all_itemsets = []
    for value in transactions.values():
        all_itemsets.extend(value)
    unique_itemsets = set(all_itemsets)

    # print(f'Original itemset: {unique_itemsets}')

    forbidden_itemset = []
    frequent_itemsets_1 = []
    for itemset in unique_itemsets:
        support = compute_support({itemset}, transactions)

        if support >= minsup:
            frequent_itemsets_1.append({itemset})
        else:
            forbidden_itemset.append({itemset})
    
    return frequent_itemsets_1, forbidden_itemset



def compute_support(itemset, transactions):
    """
    This function is supposed to compute the support for a given itemset, so:
    itemset: a Python set or frozenset of items, e.g. {'Espresso', 'Muffin'}
    transactions: dict or list of itemsets, all receipts (like your `transactions` dict)
    Computes frequency of itemset among all receipts / all receipts
    returns: support as a fraction in [0,1]
    """
    # {1: ['Espresso', 'Croissant', 'Water'], 2: ['Latte', 'Muffin'], 3: ['Espresso', 'Latte', 'Muffin'], 4: ['Cappuccino', 'Croissant'], 5: ['Espresso', 'Cappuccino', 'Brownie'], 6: ['Latte', 'Croissant', 'Water'], 7: ['Espresso', 'Muffin'], 8: ['Cappuccino', 'Muffin', 'Brownie'], 9: ['Latte', 'Brownie'], 10: ['Espresso', 'Latte', 'Croissant', 'Muffin']}
    # {'Croissant'} / {'Croissant', 'Cappuccino'}
    
    count = 0

    for receipt in transactions:
        found = True

        for item in itemset:
            
            if transactions[receipt].count(item) >= 1:          # if item in transactions[receipt]
                continue
            else:
                found = False
                break

        if found:
            count += 1
        
    # support = occurencies item / all receits
    support = count / len(transactions)
    
    return support
    
def generate_candidates(prev_freq_itemsets, k):
    """
    prev_freq_itemsets: list of frequent (k-1)-itemsets (as sets/frozensets)
    k: size of new candidate itemsets to generate
    returns: list (or set) of candidate k-itemsets
    """
    new_candidates = []
    for i in range(len(prev_freq_itemsets)):
        for j in range(i + 1, len(prev_freq_itemsets)):

            fixed_candidate = prev_freq_itemsets[i]
            varying_candidate = prev_freq_itemsets[j]

            # concatenate them --> thois weird way is how you concatenate frozenset ... bah
            new_candidate = fixed_candidate | varying_candidate

            if len(new_candidate) == k:
                new_candidates.append(new_candidate)

    return new_candidates



def get_f_2_itemsets(new_candidates_2, transactions, minsup=0.3):
    # look for [{'Croissant', 'Cappuccino'}, {'Croissant', 'Brownie'}, {'Croissant', 'Latte'}, {'Croissant', 'Muffin'}, {'Croissant', 'Espresso'},
    # in {1: ['Espresso', 'Croissant', 'Water'], 2: ['Latte', 'Muffin'], 3: ['Espresso', 'Latte', 'Muffin'], 4: ['Cappuccino', 'Croissant'], 5: ['Espresso', 'Cappuccino', 'Brownie'], 6: ['Latte', 'Croissant', 'Water'], 7: ['Espresso', 'Muffin'], 8: ['Cappuccino', 'Muffin', 'Brownie'], 9: ['Latte', 'Brownie'], 10: ['Espresso', 'Latte', 'Croissant', 'Muffin']}

    forbidden_itemset = []
    frequent_itemsets_2 = []
    for candidate in new_candidates_2:
        support = compute_support(candidate, transactions)
        
        if support >= minsup:
            frequent_itemsets_2.append(candidate)
        else:
            forbidden_itemset.append(candidate)
    
    return frequent_itemsets_2, forbidden_itemset

def get_f_3_itemsets(new_candidates_3, transactions, minsup=0.3):
    forbidden_itemset = []
    frequent_itemsets_3 = []
    for candidate in new_candidates_3:
        support = compute_support(candidate, transactions)
        
        if support >= minsup:
            frequent_itemsets_3.append(candidate)
        else:
            forbidden_itemset.append(candidate)
    
    return frequent_itemsets_3, forbidden_itemset


if __name__ == '__main__':

    df = get_data(file_path = '../../Dataset/LAB7/toy_dataset.txt')
    transactions = get_transactions(df)
    
    # construct L1:
        # generate FREQUENT all 1-itemsets --> compute support and keep only those whose support > minsup
    frequent_itemsets_1, forbidden_itemset = get_f_1_itemsets(transactions)
    print(f"Frequent 1-itemset: {frequent_itemsets_1}")
    print(f"Forbidden 1-itemsets: {forbidden_itemset}")
    print()
        
        # Generate C2 from L1, 
    new_candidates_2 = generate_candidates(frequent_itemsets_1, k=2)

        # compute support, prune to get L2.
    frequent_itemsets_2, forbidden_itemset = get_f_2_itemsets(new_candidates_2, transactions)
    print(f"Frequent 2-itemset: {frequent_itemsets_2}")
    print(f"Forbidden 2-itemsets: {forbidden_itemset}")
    print()

        # Generate C3 from L2,
    new_candidates_3 = generate_candidates(frequent_itemsets_2, k=3)
    
        # compute support, prune to get L3.  
    frequent_itemsets_3, forbidden_itemset = get_f_3_itemsets(new_candidates_3, transactions)
    print(f"Frequent 3-itemset: {frequent_itemsets_3}")
    print(f"Forbidden 3-itemsets: {forbidden_itemset}")

# while this is nice

In [ ]:
def get_data(file_path):
    df = pd.read_csv(file_path, sep=',')
    df.rename(columns={'ReceiptID': 'id', 'Item': 'item'}, inplace=True)
    return df

def get_transactions(df):
    grouped = df.groupby('id')
    transactions = {}
    for id, values in grouped:
        transactions[id] = values['item'].tolist()
    return transactions

def get_f_itemsets(k, prev_freq_itemsets, transactions, minsup=0.3):
    """
    Generate frequent k-itemsets by first generating candidates and computing their support.
    """
    # Generate Ck from L(k-1)
    new_candidates = generate_candidates(prev_freq_itemsets, k)
    
    # Compute support for new candidates, and keep those >= minsup
    frequent_itemsets, forbidden_itemset = get_f_k_itemsets(new_candidates, transactions, minsup)
    
    return frequent_itemsets, forbidden_itemset

def get_f_k_itemsets(new_candidates, transactions, minsup):
    """
    Compute support for candidate itemsets and return the frequent itemsets.
    """
    forbidden_itemset = []
    frequent_itemsets = []
    
    for candidate in new_candidates:
        support = compute_support(candidate, transactions)
        
        if support >= minsup:
            frequent_itemsets.append(candidate)
        else:
            forbidden_itemset.append(candidate)
    
    return frequent_itemsets, forbidden_itemset

def compute_support(itemset, transactions):
    """
    Computes the support for a given itemset in the transactions.
    """
    count = 0
    for receipt in transactions:
        found = True
        for item in itemset:
            if item not in transactions[receipt]:
                found = False
                break
        if found:
            count += 1
    
    # Calculate support as count / total transactions
    support = count / len(transactions)
    return support

def generate_candidates(prev_freq_itemsets, k):
    """
    Generate candidate k-itemsets from (k-1)-itemsets.
    """
    new_candidates = []

    # If prev_freq_itemsets is empty, generate 1-itemsets (this happens in the first pass)
    if not prev_freq_itemsets:
        all_itemsets = []
        for value in transactions.values():
            all_itemsets.extend(value)
        unique_itemsets = set(all_itemsets)
        for item in unique_itemsets:
            new_candidates.append(frozenset([item]))
    
    else:

        # Normal candidate generation from (k-1)-itemsets
        for i in range(len(prev_freq_itemsets)):
            for j in range(i + 1, len(prev_freq_itemsets)):
                fixed_candidate = prev_freq_itemsets[i]
                varying_candidate = prev_freq_itemsets[j]

                # Concatenate them as a frozenset
                new_candidate = fixed_candidate | varying_candidate

                if len(new_candidate) == k:
                    new_candidates.append(new_candidate)
    
    return new_candidates

if __name__ == '__main__':
    df = get_data(file_path='../../Dataset/LAB7/toy_dataset.txt')
    transactions = get_transactions(df)
    
    # Initialize L1 --> which I have to do 'manually', it cannot be done automatically --> I had to build the 'if' in the generate_candidates to handle this exception
    frequent_itemsets_1, forbidden_itemset = get_f_itemsets(1, [], transactions)
    print(f"Frequent 1-itemsets: {frequent_itemsets_1}")
    print(f"Forbidden 1-itemsets: {forbidden_itemset}")
    print()

    # Now generate L2, L3, etc. dynamically
    current_freq_itemsets = frequent_itemsets_1
    k = 2
    while current_freq_itemsets:  # Repeat until no more frequent itemsets are found
        print(f"--- Generating {k}-itemsets ---")
        
        frequent_itemsets_k, forbidden_itemset = get_f_itemsets(k, current_freq_itemsets, transactions)
        print(f"Frequent {k}-itemsets: {frequent_itemsets_k}")
        print(f"Forbidden {k}-itemsets: {forbidden_itemset}")
        print()
        
        current_freq_itemsets = frequent_itemsets_k  # Update for next iteration
        k += 1  # Increase the size for the next iteration